# Lab 6 — Convolutional layers and a first CNN

**Deep Learning for Computer Vision — Day 2** · ECAM, 5MAU · 2026–2027

**Goals.**
1. See that `nn.Conv2d` is the convolution of Day 1 — with a Sobel kernel set by hand.
2. Compute output sizes and numbers of parameters of convolutional layers (exam classic!).
3. Build and train a small CNN on Fashion-MNIST; compare it with the MLP of Lab 5.
4. Test the **translation robustness** of an MLP and of a CNN.
5. Visualise the learned filters and the feature maps.

**Duration:** ~45 min.

> *References:* Stanford CS231n, Lecture 5 (*Convolutional Neural Networks*), 2025 · KenN7, *Deep learning for computer vision*, ECAM, lab 5 (*cifar_cnn*), https://github.com/KenN7/deep-learning · V. Dumoulin, F. Visin, *A guide to convolution arithmetic for deep learning*, arXiv:1603.07285 · Y. LeCun *et al.*, "Gradient-based learning applied to document recognition", *Proc. IEEE*, 1998.

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "matplotlib": "matplotlib",
            "cv2": "opencv-contrib-python", "ipywidgets": "ipywidgets"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

DATA = Path("data") if Path("data").exists() else Path("../data")
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. `nn.Conv2d` = the convolution of Day 1

A `Conv2d(in_channels, out_channels, kernel_size)` layer holds a weight tensor of shape `(out_channels, in_channels, k, k)` and a bias of shape `(out_channels,)`. Each output channel is the sum over the input channels of a 2D correlation, plus a bias.

### Exercise 1.1
Create a `Conv2d(1, 2, 3, padding=1, bias=False)` whose two filters are Sobel-x and Sobel-y (set `conv.weight.data`), apply it to a Fashion-MNIST image and compare with `cv2.filter2D`.

In [ ]:
test_set = datasets.FashionMNIST(DATA, train=False, download=True, transform=transforms.ToTensor())
img, label = test_set[0]  # (1, 28, 28)
sobel_x = torch.tensor([[-1.0, 0, 1], [-2, 0, 2], [-1, 0, 1]])

conv = nn.Conv2d(1, 2, kernel_size=3, padding=1, bias=False)
# TODO
...
with torch.no_grad():
    out = conv(img.unsqueeze(0))  # (1, 2, 28, 28)
print("weight:", tuple(conv.weight.shape), "-> output:", tuple(out.shape))

ref = cv2.filter2D(img[0].numpy(), -1, sobel_x.numpy(), borderType=cv2.BORDER_CONSTANT)
assert np.allclose(out[0, 0].numpy(), ref, atol=1e-5)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, t in zip(ax, [img[0], out[0, 0], out[0, 1]], ["input", "filter 0 (Sobel x)", "filter 1 (Sobel y)"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")

In a CNN, these weights are **not** set by hand: they are initialised randomly and **learned** by gradient descent, like the weights of the MLP.

## 2. Shapes and parameters

For an input of size $N \times N$, kernel $K$, padding $P$, stride $S$:
$$ N_{out} = \left\lfloor \frac{N - K + 2P}{S} \right\rfloor + 1, \qquad \#\text{params} = C_{out}\,(C_{in}\,K^2 + 1). $$

### Exercise 2.1
Implement both formulas, then fill in the table **by hand first** for the network below (input `(1, 1, 28, 28)`), and check with the code.

| layer | output shape | # params |
|---|---|---|
| Conv2d(1, 16, 5, padding=2) | ? | ? |
| MaxPool2d(2) | ? | 0 |
| Conv2d(16, 32, 3, padding=1) | ? | ? |
| MaxPool2d(2) | ? | 0 |
| Flatten + Linear(?, 10) | (1, 10) | ? |

In [ ]:
def conv_out(n, k, p=0, s=1):
    # TODO
    ...


def conv_params(c_in, c_out, k, bias=True):
    # TODO
    ...


assert conv_out(28, 5, p=2) == 28 and conv_out(32, 3, s=2) == 15 and conv_out(224, 7, p=3, s=2) == 112
assert conv_params(3, 64, 3) == 1792


def summary(model, input_shape):
    """Print the output shape and the number of parameters of each layer (forward hooks)."""
    rows = []
    hooks = [m.register_forward_hook(lambda m, i, o: rows.append((m, tuple(o.shape))))
             for m in model.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        model(torch.zeros(input_shape))
    for h in hooks:
        h.remove()
    for m, shape in rows:
        print(f"{m.__class__.__name__:12s} {str(shape):18s} {sum(p.numel() for p in m.parameters()):8d}")
    print("total parameters:", sum(p.numel() for p in model.parameters()))


net = nn.Sequential(nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
summary(net, (1, 1, 28, 28))

## 3. A first CNN on Fashion-MNIST

### Exercise 3.1
Complete `SmallCNN`: two blocks *conv → ReLU → max-pool* (16 then 32 channels, 3×3 kernels, padding 1), then a classifier *flatten → linear(…, 64) → ReLU → linear(64, 10)*. Then train it and compare with the MLP (copied from Lab 5).

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        ...

    def forward(self, x):
        return self.classifier(self.features(x))


summary(SmallCNN(), (1, 1, 28, 28))
assert SmallCNN()(torch.zeros(2, 1, 28, 28)).shape == (2, 10)

In [ ]:
tfm = transforms.ToTensor()
train_full = datasets.FashionMNIST(DATA, train=True, download=True, transform=tfm)
train_set, val_set = torch.utils.data.random_split(train_full, [50000, 10000],
                                                   generator=torch.Generator().manual_seed(0))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=512)
classes = train_full.classes


def run_epoch(model, loader, optimizer=None):
    """One pass over the loader; trains if an optimizer is given. Returns (loss, accuracy)."""
    train = optimizer is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total / n, correct / n


def fit(model, epochs=3, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(epochs):
        t0 = time.perf_counter()
        tl, ta = run_epoch(model, train_loader, opt)
        vl, va = run_epoch(model, val_loader)
        print(f"epoch {e + 1}: train acc {ta:.3f} | val acc {va:.3f} ({time.perf_counter() - t0:.1f}s)")
    return va


torch.manual_seed(0)
mlp = MLP()
acc_mlp = fit(mlp)
torch.manual_seed(0)
cnn = SmallCNN()
acc_cnn = fit(cnn)
print(f"\nMLP {acc_mlp:.3f} with {sum(p.numel() for p in mlp.parameters())} params | "
      f"CNN {acc_cnn:.3f} with {sum(p.numel() for p in cnn.parameters())} params")

## 4. Translation: MLP vs CNN

A conveyor belt never puts the part exactly at the same place. Shift the validation images by a few pixels and measure the accuracy of both models (`torch.roll` shifts with wrap-around; the borders of Fashion-MNIST are black, so it is a clean shift).

### Exercise 4.1
Complete `accuracy_shifted(model, dx)` and plot the accuracy vs the shift for both models. Explain the difference.

In [ ]:
@torch.no_grad()
def accuracy_shifted(model, dx):
    model.eval()
    correct, n = 0, 0
    for x, y in val_loader:
        # TODO
        ...
    return correct / n


shifts = [0, 1, 2, 3, 4, 5, 6]
acc_m = [accuracy_shifted(mlp, s) for s in shifts]
acc_c = [accuracy_shifted(cnn, s) for s in shifts]
plt.plot(shifts, acc_m, "o-", label="MLP"); plt.plot(shifts, acc_c, "o-", label="CNN")
plt.xlabel("horizontal shift [pixels]"); plt.ylabel("val accuracy"); plt.legend(); plt.grid(alpha=.3);
assert acc_c[4] > acc_m[4]

## 5. What did the CNN learn?

### Exercise 5.1 — Filters
Display the 16 filters of the first convolution (`cnn.features[0].weight`, shape `(16, 1, 3, 3)`). Do you recognise kernels from Day 1?

In [ ]:
w = cnn.features[0].weight.detach().cpu()
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for k, a in enumerate(ax.ravel()):
    a.imshow(w[k, 0], cmap="bwr", vmin=-w.abs().max(), vmax=w.abs().max()); a.axis("off")
plt.suptitle("first-layer filters (red > 0, blue < 0)");

### Exercise 5.2 — Feature maps
Display the 16 output channels of the first block for one image, and the 32 of the second block. How do they change with depth?

In [ ]:
x = img.unsqueeze(0).to(device)
with torch.no_grad():
    # TODO
    f1 = ... ; f2 = ...
fig, ax = plt.subplots(3, 16, figsize=(16, 3.4))
ax[0, 0].imshow(img[0], cmap="gray")
for k in range(16):
    ax[1, k].imshow(f1[k], cmap="viridis")
    ax[2, k].imshow(f2[2 * k], cmap="viridis")
for a in ax.ravel():
    a.axis("off")
ax[0, 0].set_title(classes[label], fontsize=8);

## 6. (Bonus) Your own CNN on CIFAR-10

CIFAR-10 (60 000 colour images 32×32, 10 classes, ≈ 170 MB download) is much harder than Fashion-MNIST. If you have time (and bandwidth), adapt `SmallCNN` to 3 input channels and 32×32 images (what changes in the classifier?) and train it for a few epochs. Expect ~60–70 % accuracy for a small CNN.

In [ ]:
RUN_CIFAR = False
if RUN_CIFAR:
    cifar = datasets.CIFAR10(DATA, train=True, download=True, transform=transforms.ToTensor())
    # ... your code here